Environment Set-Up

In [2]:
# Library Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

In [5]:
# Accessibility - to be applied to all plots
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False
})

# Colorblind safe palette
sns.set_palette("colorblind")

print("Libraries loaded and styling applied successfully")


Libraries loaded and styling applied successfully


Loading the Dataset

In [18]:
# Install the UCI ML repository package
!pip install ucimlrepo

from ucimlrepo import fetch_ucirepo

# Fetch official UCI Heart Disease dataset
heart_disease = fetch_ucirepo(id=45)

# Separate features and target
X = heart_disease.data.features
y = heart_disease.data.targets

# Combine into single dataframe
UCI = pd.concat([X, y], axis=1)

# Check
print("Dataset Shape:", UCI.shape)
print("\nColumn Names:")
print(UCI.columns.tolist())
print("\nFirst 5 Rows:")
UCI.head()

Dataset Shape: (303, 14)

Column Names:
['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']

First 5 Rows:


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,2
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


Data Inspection

In [19]:
# Check original num values
print("Original num value counts:")
print(UCI['num'].value_counts().sort_index())

# Convert to binary: 0 = no disease, 1 = disease present
UCI['target'] = (UCI['num'] > 0).astype(int)

# Drop original num column
UCI = UCI.drop(columns=['num'])

# Verify conversion
print("\nConverted target value counts:")
print(UCI['target'].value_counts())

print("\nTarget proportions:")
print(UCI['target'].value_counts(normalize=True).round(3))

print("\nNew column list:")
print(UCI.columns.tolist())

Original num value counts:
num
0    164
1     55
2     36
3     35
4     13
Name: count, dtype: int64

Converted target value counts:
target
0    164
1    139
Name: count, dtype: int64

Target proportions:
target
0    0.541
1    0.459
Name: proportion, dtype: float64

New column list:
['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']


In [20]:
# Check for missing values
print("Data Types:")
print(UCI.dtypes)

print("\nMissing Values Per Column:")
print(UCI.isnull().sum())

print("\nTotal missing values:", UCI.isnull().sum().sum())

print("\nPercentage missing per column:")
print((UCI.isnull().sum() / len(UCI) * 100).round(2))

Data Types:
age           int64
sex           int64
cp            int64
trestbps      int64
chol          int64
fbs           int64
restecg       int64
thalach       int64
exang         int64
oldpeak     float64
slope         int64
ca          float64
thal        float64
target        int64
dtype: object

Missing Values Per Column:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
target      0
dtype: int64

Total missing values: 6

Percentage missing per column:
age         0.00
sex         0.00
cp          0.00
trestbps    0.00
chol        0.00
fbs         0.00
restecg     0.00
thalach     0.00
exang       0.00
oldpeak     0.00
slope       0.00
ca          1.32
thal        0.66
target      0.00
dtype: float64


In [21]:
# Uniqueness for Categorical Variables
categorical = ['sex', 'cp', 'fbs', 'restecg', 'exang',
               'slope', 'ca', 'thal', 'target']

continuous = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

for col in categorical:
    print(f"{col}: {sorted(UCI[col].dropna().unique())} "
          f"({UCI[col].nunique()} unique values) "
          f"| Missing: {UCI[col].isnull().sum()}")

sex: [np.int64(0), np.int64(1)] (2 unique values) | Missing: 0
cp: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)] (4 unique values) | Missing: 0
fbs: [np.int64(0), np.int64(1)] (2 unique values) | Missing: 0
restecg: [np.int64(0), np.int64(1), np.int64(2)] (3 unique values) | Missing: 0
exang: [np.int64(0), np.int64(1)] (2 unique values) | Missing: 0
slope: [np.int64(1), np.int64(2), np.int64(3)] (3 unique values) | Missing: 0
ca: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0)] (4 unique values) | Missing: 4
thal: [np.float64(3.0), np.float64(6.0), np.float64(7.0)] (3 unique values) | Missing: 2
target: [np.int64(0), np.int64(1)] (2 unique values) | Missing: 0


In [22]:
# Descriptive Statistics for Continuous Variables
cont_stats = UCI[continuous].describe().T

# Add skewness and kurtosis
cont_stats['skewness'] = UCI[continuous].skew()
cont_stats['kurtosis'] = UCI[continuous].kurt()

print("Descriptive Statistics - Continuous Variables:")
print(cont_stats.round(2))

Descriptive Statistics - Continuous Variables:
          count    mean    std    min    25%    50%    75%    max  skewness  \
age       303.0   54.44   9.04   29.0   48.0   56.0   61.0   77.0     -0.21   
trestbps  303.0  131.69  17.60   94.0  120.0  130.0  140.0  200.0      0.71   
chol      303.0  246.69  51.78  126.0  211.0  241.0  275.0  564.0      1.14   
thalach   303.0  149.61  22.88   71.0  133.5  153.0  166.0  202.0     -0.54   
oldpeak   303.0    1.04   1.16    0.0    0.0    0.8    1.6    6.2      1.27   

          kurtosis  
age          -0.52  
trestbps      0.88  
chol          4.49  
thalach      -0.05  
oldpeak       1.58  
